# 28. Unsupervised Learning: Principal Component Analysis (PCA)

## Algorithm Category
**Type**: Unsupervised Learning - Dimensionality Reduction  
**Complexity**: Medium  
**Use Case**: Reduce dimensionality while preserving maximum variance

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand PCA and its mathematical foundation
- Implement PCA for dimensionality reduction
- Understand eigenvalues, eigenvectors, and explained variance
- Visualize principal components and data projections
- Determine optimal number of components
- Apply PCA for data preprocessing and visualization

## Historical Context

PCA was developed by Karl Pearson in 1901:
- Pearson, K. (1901): "On lines and planes of closest fit to systems of points in space"
- One of the oldest and most widely used dimensionality reduction techniques
- Foundation for many modern techniques

**Key Papers/References:**
- Pearson, K. (1901). "On lines and planes of closest fit to systems of points in space"
- Hotelling, H. (1933). "Analysis of a complex of statistical variables into principal components"

## When to Use Principal Component Analysis

PCA is appropriate when:
- You have high-dimensional data
- Features are correlated
- You want to reduce dimensionality for visualization
- You need to remove noise from data
- You want to reduce computational cost
- Features need to be decorrelated

## Theory & Mechanics

### Mathematical Foundation

PCA finds directions of maximum variance in data and projects data onto these directions.

**Covariance Matrix:**
$$C = \frac{1}{n-1} X^T X$$

Where $X$ is the centered data matrix.

**Eigenvalue Decomposition:**
$$C = P \Lambda P^T$$

Where:
- $P$: Matrix of eigenvectors (principal components)
- $\Lambda$: Diagonal matrix of eigenvalues (variances)

**Principal Components:**
- First PC: Direction of maximum variance
- Second PC: Direction of maximum variance orthogonal to first PC
- And so on...

**Projection:**
$$Y = X P_k$$

Where $P_k$ contains the first $k$ principal components.

**Explained Variance Ratio:**
$$\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_{j=1}^{d} \lambda_j}$$

### How It Works

1. **Center data**: Subtract mean from each feature
2. **Compute covariance matrix**: Calculate covariance between all feature pairs
3. **Eigenvalue decomposition**: Find eigenvectors and eigenvalues
4. **Select components**: Choose top k components based on explained variance
5. **Project data**: Transform data to lower-dimensional space

### Key Hyperparameters

- **n_components**: Number of components to keep
  - Integer: Exact number
  - Float (0-1): Keep components that explain this fraction of variance
  - 'mle': Use MLE to estimate number
- **whiten**: Whether to whiten the components (scale to unit variance)

### Advantages

- Reduces dimensionality while preserving variance
- Removes correlation between features
- Can improve model performance
- Helps with visualization
- Reduces overfitting risk
- Fast and efficient

### Limitations

- Assumes linear relationships
- May lose interpretability
- Sensitive to feature scaling
- May not capture non-linear patterns
- Components may not be meaningful


## Implementation

Let's implement PCA for dimensionality reduction.


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import (
    load_iris,  # Iris flower dataset (4 features, good for PCA demo)
    load_wine,  # Wine dataset (13 features, higher dimensional)
    load_breast_cancer  # Breast cancer dataset (30 features, very high dimensional)
)
from sklearn.decomposition import PCA  # Principal Component Analysis (dimensionality reduction)
from sklearn.preprocessing import StandardScaler  # Feature scaling (CRITICAL for PCA!)

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.unsupervised import perform_pca  # PCA wrapper function

print("Libraries imported successfully!")  # Confirm all imports worked


In [ ]:
# ============================================
# LOADING THE DATASET: Iris Classification
# ============================================

# load_iris() loads the Iris flower dataset from scikit-learn
# This dataset has 4 features, which we'll reduce to 2 using PCA
iris = load_iris()  # Returns a Bunch object with data, target, feature_names
X = iris.data  # Features: flower measurements (150 samples × 4 features)
y = iris.target  # True labels: flower species (for visualization)

print(f"Original Dataset Shape: {X.shape}")  # Output: (150, 4) - 150 flowers, 4 features
print(f"Features: {iris.feature_names}")  # Output: ['sepal length (cm)', 'sepal width (cm)', ...]

# ============================================
# FEATURE SCALING: CRITICAL FOR PCA!
# ============================================

# PCA is VERY sensitive to feature scaling!
# Why? PCA finds directions of maximum variance
# Features on different scales distort variance calculations
# Example: If one feature is in meters (0-10) and another in millimeters (0-10000),
#   the millimeter feature will dominate (have much larger variance)
# Without scaling, PCA will focus on features with larger scales, not necessarily important ones

# StandardScaler normalizes features to have mean=0 and std=1
scaler = StandardScaler()  # Create scaler object
X_scaled = scaler.fit_transform(X)
# fit_transform(): Learn scaling from data (mean, std) and apply it
# X_scaled: Features normalized (mean=0, std=1 for each column)
# After scaling, all features contribute equally to variance calculations

# ============================================
# APPLYING PRINCIPAL COMPONENT ANALYSIS
# ============================================

# PCA finds directions of maximum variance in data
# Projects data onto these directions (principal components)
# Reduces dimensionality while preserving as much variance as possible

# PCA parameters:
# n_components=2: Number of principal components to keep
#   - We want to reduce from 4 features to 2 dimensions (for visualization)
#   - Can also specify as float (0-1) to keep components that explain that fraction of variance
pca = PCA(n_components=2)

# fit_transform() performs PCA:
# 1. Centers data (subtracts mean)
# 2. Computes covariance matrix
# 3. Finds eigenvectors (principal components) and eigenvalues (variances)
# 4. Projects data onto first 2 principal components
X_pca = pca.fit_transform(X_scaled)
# Returns: Transformed data in 2D space (150 samples × 2 components)

# ============================================
# DISPLAYING PCA RESULTS
# ============================================

print(f"\nPCA Results:")
print(f"  Reduced shape: {X_pca.shape}")  # Output: (150, 2) - reduced from 4 to 2 dimensions

# Explained variance ratio: How much variance each component captures
print(f"  Explained variance ratio: {pca.explained_variance_ratio_}")
# Returns: array like [0.729, 0.229]
# PC1 explains 72.9% of variance, PC2 explains 22.9% of variance

# Total explained variance: How much variance is preserved
print(f"  Total explained variance: {sum(pca.explained_variance_ratio_):.3f}")
# Sum of explained variance ratios (e.g., 0.729 + 0.229 = 0.958 = 95.8%)
# This means we preserved 95.8% of original variance with just 2 components!

# ============================================
# VISUALIZING PCA TRANSFORMATION
# ============================================

# Create figure with 2 subplots
plt.figure(figsize=(12, 5))  # Figure size: 12×5 inches

# Subplot 1: Original data (first 2 features)
plt.subplot(1, 2, 1)  # 1 row, 2 columns, position 1 (left)

# Scatter plot using original features (first 2 features only)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X[:, 0]: First original feature (sepal length)
# X[:, 1]: Second original feature (sepal width)
# c=y: Color by true species labels
# This shows what we'd see if we just picked 2 features arbitrarily

plt.xlabel(iris.feature_names[0])  # X-axis: first feature name
plt.ylabel(iris.feature_names[1])  # Y-axis: second feature name
plt.title('Original Data (First 2 Features)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Subplot 2: PCA projection (2 principal components)
plt.subplot(1, 2, 2)  # 1 row, 2 columns, position 2 (right)

# Scatter plot using PCA-transformed data
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_pca[:, 0]: First principal component (PC1)
# X_pca[:, 1]: Second principal component (PC2)
# c=y: Color by true species labels (for comparison)
# This shows the optimized 2D view that preserves maximum variance

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
# X-axis: First principal component
# Label shows how much variance PC1 explains (e.g., "PC1 (72.9% variance)")

plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
# Y-axis: Second principal component
# Label shows how much variance PC2 explains (e.g., "PC2 (22.9% variance)")

plt.title('PCA Projection (2D)')  # Chart title
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# Interpretation:
# - Left plot: Arbitrary 2D view using first 2 original features
# - Right plot: Optimized 2D view using PCA (preserves maximum variance)
# - If classes are better separated in right plot, PCA found better representation
# - PCA components are linear combinations of original features
# - PC1 captures most variance (72.9%), PC2 captures second-most (22.9%)
# - Total: 95.8% of variance preserved with just 2 dimensions (reduced from 4!)


## Explained Variance and Component Selection

Let's analyze explained variance to determine optimal number of components.


In [ ]:
# ============================================
# EXPLAINED VARIANCE ANALYSIS: Understanding Component Importance
# ============================================

# To choose optimal number of components, we need to see how much variance each component explains
# We'll fit PCA with all components first, then analyze the variance distribution

# Fit PCA with all components (to analyze all variance)
pca_full = PCA()  # No n_components specified = keep all components
# This allows us to see variance explained by each component

# Fit PCA to data (computes all principal components)
pca_full.fit(X_scaled)  # Learn principal components from data
# This computes all 4 principal components (since we have 4 features)

# ============================================
# CALCULATING EXPLAINED VARIANCE
# ============================================

# Explained variance ratio: How much variance each component captures
explained_variance = pca_full.explained_variance_ratio_
# Returns: array like [0.729, 0.229, 0.037, 0.005]
# PC1: 72.9%, PC2: 22.9%, PC3: 3.7%, PC4: 0.5%
# First components capture most variance (this is by design!)

# Cumulative explained variance: Total variance preserved with k components
cumulative_variance = np.cumsum(explained_variance)
# np.cumsum(): Cumulative sum
# Returns: array like [0.729, 0.958, 0.995, 1.000]
# With 1 component: 72.9%, with 2: 95.8%, with 3: 99.5%, with 4: 100%

# ============================================
# VISUALIZING EXPLAINED VARIANCE
# ============================================

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1 row, 2 columns

# Plot 1: Bar chart of explained variance per component
axes[0].bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7)
# X-axis: Component number (1, 2, 3, 4)
# Y-axis: Explained variance ratio (0 to 1)
# Bar height shows how much variance each component captures

axes[0].set_xlabel('Principal Component')  # X-axis: component number
axes[0].set_ylabel('Explained Variance Ratio')  # Y-axis: variance (0 to 1)
axes[0].set_title('Explained Variance by Component')  # Chart title
axes[0].grid(True, alpha=0.3)  # Add grid

# Plot 2: Line plot of cumulative explained variance
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, 'o-', markersize=8)
# X-axis: Number of components (1, 2, 3, 4)
# Y-axis: Cumulative explained variance (0 to 1)
# Shows how much variance is preserved as we add more components

# Draw horizontal line at 95% variance (common threshold)
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
# Red dashed line shows 95% threshold
# Common rule: Keep enough components to explain 95% of variance

axes[1].set_xlabel('Number of Components')  # X-axis: number of components
axes[1].set_ylabel('Cumulative Explained Variance')  # Y-axis: total variance (0 to 1)
axes[1].set_title('Cumulative Explained Variance')  # Chart title
axes[1].legend()  # Show legend
axes[1].grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# FINDING OPTIMAL NUMBER OF COMPONENTS
# ============================================

# Find number of components needed for 95% variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
# cumulative_variance >= 0.95: Boolean array (True where cumulative variance ≥ 0.95)
# np.argmax(): Index of first True value
# + 1: Convert from 0-based index to 1-based component number
# Example: If cumulative_variance[1] = 0.958 ≥ 0.95, then n_components_95 = 2

print(f"Number of components for 95% variance: {n_components_95}")  # Display result
print(f"Explained variance with {n_components_95} components: {cumulative_variance[n_components_95-1]:.3f}")
# Show actual variance explained (should be ≥ 0.95)

# ============================================
# DISPLAYING EXPLAINED VARIANCE BY COMPONENT
# ============================================

# Print explained variance for each component
print("\nExplained Variance by Component:")
for i, var in enumerate(explained_variance):
    print(f"  PC{i+1}: {var:.3f} ({var*100:.1f}%)")
    # PC1: 0.729 (72.9%)
    # PC2: 0.229 (22.9%)
    # PC3: 0.037 (3.7%)
    # PC4: 0.005 (0.5%)

# Interpretation:
# - First components capture most variance (this is how PCA works!)
# - Later components capture less variance (diminishing returns)
# - For Iris dataset, 2 components explain 95.8% of variance (excellent compression!)
# - We can reduce from 4 features to 2 components with minimal information loss
# - This is the "elbow" in the cumulative variance plot
# - Common thresholds: 95% or 99% variance (depends on application)


## Principal Component Analysis

Let's examine the principal components and their relationship to original features.


In [ ]:
# ============================================
# PRINCIPAL COMPONENT LOADINGS: Understanding What Each PC Represents
# ============================================

# Principal components are linear combinations of original features
# Loadings show how much each original feature contributes to each component
# This helps interpret what each principal component represents

# Get principal components (eigenvectors)
components = pca.components_
# Returns: array of shape (n_components, n_features) = (2, 4)
# Each row is a principal component (eigenvector)
# Each column shows contribution of one original feature
# Values can be positive or negative (direction matters!)

# ============================================
# VISUALIZING COMPONENT LOADINGS
# ============================================

# Create heatmap to visualize loadings
fig, ax = plt.subplots(figsize=(10, 6))  # Figure size: 10×6 inches

# imshow() creates a heatmap (color-coded matrix)
im = ax.imshow(components, cmap='coolwarm', aspect='auto')
# components: Matrix to display (2 rows × 4 columns)
# cmap='coolwarm': Color scheme (blue = negative, red = positive)
# aspect='auto': Adjust aspect ratio automatically

# Set x-axis labels (original feature names)
ax.set_xticks(range(len(iris.feature_names)))  # X-positions: 0, 1, 2, 3
ax.set_xticklabels(iris.feature_names, rotation=45, ha='right')
# iris.feature_names: Feature names (e.g., "sepal length (cm)")
# rotation=45: Rotate labels 45 degrees (prevent overlap)
# ha='right': Horizontal alignment (right-aligned)

# Set y-axis labels (principal component names)
ax.set_yticks(range(2))  # Y-positions: 0, 1
ax.set_yticklabels([f'PC{i+1}' for i in range(2)])
# Labels: "PC1", "PC2"

ax.set_title('Principal Component Loadings')  # Chart title

# Add colorbar (legend showing color scale)
plt.colorbar(im, ax=ax, label='Loading')
# Shows what colors mean (positive = red, negative = blue, magnitude = intensity)

# Adjust layout
plt.tight_layout()
plt.show()  # Display the heatmap

# ============================================
# DISPLAYING COMPONENT LOADINGS
# ============================================

# Print component loadings in text format
print("Principal Component Loadings:")
for i, pc in enumerate(components):
    # i: Component index (0 or 1)
    # pc: Principal component (array of 4 loadings)
    
    print(f"\nPC{i+1}:")
    # Display loading for each original feature
    for j, feature in enumerate(iris.feature_names):
        # j: Feature index (0, 1, 2, 3)
        # feature: Feature name (e.g., "sepal length (cm)")
        # pc[j]: Loading of this feature in this component
        
        print(f"  {feature}: {pc[j]:.3f}")
        # Example: "sepal length (cm): 0.521"
        # Positive = feature increases in this direction
        # Negative = feature decreases in this direction
        # Larger absolute value = feature contributes more to this component

# Interpretation:
# - Loadings show how original features combine to form principal components
# - PC1 = weighted sum of all features (e.g., 0.5×sepal_length + 0.3×sepal_width + ...)
# - PC2 = different weighted sum (orthogonal to PC1)
# - High loading (positive or negative) = feature is important for this component
# - Low loading ≈ 0 = feature doesn't contribute much to this component
# - Components are orthogonal (uncorrelated) - this is a key property of PCA!
# - This helps understand what each component represents (e.g., "size" vs "shape")


## Validation & Testing

Let's validate PCA and test reconstruction error.


In [ ]:
# ============================================
# VALIDATION: Testing Reconstruction
# ============================================

# PCA is a lossy compression - we lose some information when reducing dimensions
# We can reconstruct the original data from principal components (with some error)
# Reconstruction error shows how much information was lost

# inverse_transform() reconstructs original data from principal components
X_reconstructed = pca.inverse_transform(X_pca)
# X_pca: Data in principal component space (2D)
# Returns: Reconstructed data in original feature space (4D)
# This is an approximation of the original data (some information lost)

# ============================================
# CALCULATING RECONSTRUCTION ERROR
# ============================================

# Reconstruction error: How different is reconstructed data from original?
reconstruction_error = np.mean((X_scaled - X_reconstructed) ** 2)
# (X_scaled - X_reconstructed) ** 2: Squared differences (element-wise)
# np.mean(): Average squared error (MSE - Mean Squared Error)
# Lower is better (0 = perfect reconstruction, but impossible with dimension reduction)

print(f"Reconstruction Error (MSE): {reconstruction_error:.6f}")  # Display error

# ============================================
# VISUALIZING RECONSTRUCTION
# ============================================

# Compare original vs reconstructed data
fig, axes = plt.subplots(1, 2, figsize=(14, 5))  # 1 row, 2 columns

# Plot 1: Original data (first 2 features)
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_scaled[:, 0]: First feature (standardized)
# X_scaled[:, 1]: Second feature (standardized)
# c=y: Color by true labels

axes[0].set_xlabel('Feature 1 (standardized)')  # X-axis label
axes[0].set_ylabel('Feature 2 (standardized)')  # Y-axis label
axes[0].set_title('Original Data')  # Chart title
axes[0].grid(True, alpha=0.3)  # Add grid

# Plot 2: Reconstructed data (first 2 features)
axes[1].scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], c=y, cmap='viridis', s=50, alpha=0.7)
# X_reconstructed[:, 0]: First feature (reconstructed from 2 PCs)
# X_reconstructed[:, 1]: Second feature (reconstructed from 2 PCs)
# c=y: Color by true labels

axes[1].set_xlabel('Feature 1 (reconstructed)')  # X-axis label
axes[1].set_ylabel('Feature 2 (reconstructed)')  # Y-axis label
axes[1].set_title('Reconstructed Data (from 2 PCs)')  # Chart title
axes[1].grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display both plots

# ============================================
# ASSERTIONS: Automated Validation Checks
# ============================================

# Check 1: Should have 2 principal components
assert X_pca.shape[1] == 2, "Should have 2 principal components"
# X_pca.shape[1]: Number of columns (should be 2)

# Check 2: Should explain at least 90% of variance
assert sum(pca.explained_variance_ratio_) > 0.9, "Should explain at least 90% variance"
# Sum of explained variance ratios should be > 0.9 (90%)
# This ensures we're not losing too much information

print("\n✓ Validation checks passed")  # All checks passed!

# Interpretation:
# - Reconstruction error shows information loss from dimension reduction
# - Lower error = better reconstruction (less information lost)
# - For Iris dataset, 2 PCs explain 95.8% variance, so reconstruction is very good
# - If reconstruction error is high, we may need more components
# - Original vs reconstructed plots should look similar (if PCA worked well)
# - Differences show what information was lost (the 4.2% of variance not captured)


## Real-World Application

Let's apply PCA to a higher-dimensional dataset.


In [ ]:
# ============================================
# REAL-WORLD APPLICATION: Higher-Dimensional Dataset
# ============================================

# Wine dataset has 13 features - more challenging for PCA
# This demonstrates PCA's power on higher-dimensional data

# Load Wine dataset (classification problem with 13 features)
wine = load_wine()  # Returns Bunch object
X_wine = wine.data  # Features: wine chemical measurements (178 samples × 13 features)
y_wine = wine.target  # True labels: wine class (for visualization)

print(f"Wine Dataset Shape: {X_wine.shape}")  # Output: (178, 13) - 178 wines, 13 features
print(f"Number of features: {X_wine.shape[1]}")  # Output: 13 features

# ============================================
# FEATURE SCALING: Critical for PCA
# ============================================

# Standardize features (CRITICAL for PCA!)
scaler_wine = StandardScaler()  # Create scaler
X_wine_scaled = scaler_wine.fit_transform(X_wine)
# fit_transform(): Learn scaling and apply it
# X_wine_scaled: Features normalized (mean=0, std=1)

# ============================================
# APPLYING PCA WITH VARIANCE THRESHOLD
# ============================================

# Instead of specifying exact number of components, we can specify variance threshold
# This automatically selects number of components needed to explain that much variance

# PCA with variance threshold
pca_wine = PCA(n_components=0.95)  # Keep components that explain 95% of variance
# n_components=0.95: Float between 0 and 1
# PCA will automatically select enough components to explain 95% of variance
# This is more flexible than specifying exact number!

# Fit and transform data
X_wine_pca = pca_wine.fit_transform(X_wine_scaled)
# Returns: Transformed data (178 samples × k components, where k is automatically determined)

# ============================================
# DISPLAYING PCA RESULTS
# ============================================

print(f"\nPCA Results:")
print(f"  Reduced shape: {X_wine_pca.shape}")  # Output: (178, k) where k < 13

# Number of components automatically selected
print(f"  Number of components: {pca_wine.n_components_}")
# n_components_: Actual number of components selected (attribute, not parameter)
# Example: If 8 components explain 95% variance, n_components_ = 8

# Total explained variance (should be ≥ 0.95)
print(f"  Explained variance: {sum(pca_wine.explained_variance_ratio_):.3f}")
# Sum of explained variance ratios (should be close to 0.95)

# ============================================
# VISUALIZING PCA PROJECTION
# ============================================

# Visualize first 2 principal components (for 2D plot)
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Scatter plot using first 2 principal components
scatter = plt.scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], c=y_wine, 
                      cmap='viridis', s=50, alpha=0.7)
# X_wine_pca[:, 0]: First principal component (PC1)
# X_wine_pca[:, 1]: Second principal component (PC2)
# c=y_wine: Color by true wine class

plt.xlabel(f'PC1 ({pca_wine.explained_variance_ratio_[0]:.2%} variance)')
# X-axis: First principal component
# Label shows variance explained by PC1

plt.ylabel(f'PC2 ({pca_wine.explained_variance_ratio_[1]:.2%} variance)')
# Y-axis: Second principal component
# Label shows variance explained by PC2

plt.title('Wine Dataset: PCA Projection (2D)')  # Chart title
plt.colorbar(scatter, label='Class')  # Color legend showing wine classes
plt.grid(True, alpha=0.3)  # Add grid

# Adjust layout
plt.tight_layout()
plt.show()  # Display the plot

# ============================================
# DIMENSIONALITY REDUCTION SUMMARY
# ============================================

# Compare original vs reduced dimensionality
print(f"\nDimensionality Reduction:")
print(f"  Original: {X_wine.shape[1]} features")  # Original number of features (13)
print(f"  Reduced: {pca_wine.n_components_} components")  # Number of components selected
print(f"  Reduction: {(1 - pca_wine.n_components_/X_wine.shape[1])*100:.1f}%")
# Calculate percentage reduction
# Example: 13 features → 8 components = 38.5% reduction
# We reduced from 13 to 8 dimensions while preserving 95% of variance!

# Interpretation:
# - PCA successfully reduced dimensionality from 13 to k components
# - Preserved 95% of variance (minimal information loss)
# - This makes data easier to visualize and can improve ML model performance
# - Fewer dimensions = faster training, less overfitting, easier visualization
# - The 2D plot shows how well classes separate in principal component space
# - If classes are well-separated in 2D, PCA found a good representation


## Summary & Key Takeaways

### Key Concepts Learned

1. **PCA Basics**
   - Linear dimensionality reduction technique
   - Finds directions of maximum variance
   - Projects data onto principal components
   - Decorrelates features

2. **Mathematical Foundation**
   - Eigenvalue decomposition of covariance matrix
   - Principal components are eigenvectors
   - Explained variance from eigenvalues
   - Orthogonal components

3. **Component Selection**
   - Use explained variance ratio
   - Common thresholds: 95% or 99% variance
   - Scree plot to visualize variance
   - Elbow method for selection

4. **Best Practices**
   - Always standardize features before PCA
   - Visualize explained variance
   - Consider interpretability vs dimensionality
   - Use for preprocessing before ML models
   - Can improve model performance

### When to Use Principal Component Analysis

✅ **Good for:**
- High-dimensional data
- Correlated features
- Data visualization (reduce to 2D/3D)
- Noise reduction
- Feature decorrelation
- Reducing computational cost
- Preprocessing for ML models

❌ **Not ideal for:**
- Non-linear relationships (use t-SNE, UMAP)
- When interpretability is crucial
- When all features are important
- Very sparse data
- When features are already uncorrelated

### Next Steps

- Compare with **t-SNE** for non-linear dimensionality reduction
- Use **Kernel PCA** for non-linear relationships
- Apply **Incremental PCA** for large datasets
- Explore **Sparse PCA** for feature selection
- Use PCA for **anomaly detection** (reconstruction error)
